# 05 — Elo rating

Adds an Elo rating as a feature: one number per fighter, corrected after every
fight, so it is the only feature that knows how good a fighter's opponents
were (`src/ufc_predictor/features/elo.py`).

Three things were tested before any of this code was written, and they shape
the notebook:

- **K is chosen by how much the rating helps the model**, not by how well Elo
  forecasts on its own. The K that forecasts best alone (≈63) added nothing to
  the model; a faster rating (K ≈ 150–300) helped in 8 of 8 model seeds.
- **Every comparison is averaged over several model seeds.** The model looks
  at a random 43% of features at each split, so the same settings score
  differently from seed to seed (spread ≈ 0.001). The first tuning run's
  0.6705 was the luckiest of 8 seeds.
- **`model_params.json` had been overwritten by a test** (fixed in
  `data/io.py`), so the settings are re-tuned here rather than trusted.

Order: tune without Elo → Elo on its own (reference) → choose K → rebuild the
table with Elo → re-tune with Elo → one look at validation. The test years
stay closed. Runtime: about an hour, almost all of it the two tuning runs.

In [1]:
import numpy as np
import optuna
import pandas as pd

optuna.logging.set_verbosity(optuna.logging.WARNING)   # one line per trial would bury the results

from ufc_predictor.data.io import load_fighters, load_processed, save_processed
from ufc_predictor.features import manifest
from ufc_predictor.features.elo import ELO_COLUMNS, elo_forecast_log_loss, save_elo_params
from ufc_predictor.features.pipeline import build_featured_table
from ufc_predictor.features.shrinkage import load_alphas
from ufc_predictor.modeling.evaluate import compare, group_importance, is_womens_fight, scorecard
from ufc_predictor.modeling.model import fit_model, predict_proba
from ufc_predictor.modeling.split import chronological_split
from ufc_predictor.modeling.tune import choose_elo_k, cv_log_loss, save_params, tune, tuning_table

fights = load_processed("fights")
fighters = load_fighters()
alphas = load_alphas()

without_elo = build_featured_table(fights, fighters, alphas, elo_k=None)
train_without, validation_without, _ = chronological_split(without_elo)
print(f"{without_elo.shape[0]:,} fights x {without_elo.shape[1]} columns | training fights: {len(train_without):,}")

7,840 fights x 370 columns | training fights: 2,862


## 1. Tune the model without Elo

Gives the "without Elo" settings that the K search and the final comparison
need, and restores `model_params.json` honestly. Same search as before (60
Optuna trials, seeded), but each trial is now averaged over 3 model seeds so
the search can't reward a lucky draw.

In [2]:
study_without = tune(train_without, n_trials=60)
save_params(study_without)
params_without = study_without.best_params
print(f"best CV log loss, averaged over 3 seeds: {study_without.best_value:.5f}")
tuning_table(study_without, top=5)

best CV log loss, averaged over 3 seeds: 0.67170


,trial,log_loss,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features
0,37,0.67170,0.01606,150,15,30,0.52016,0.30321
1,34,0.67182,0.02488,100,21,30,0.28753,0.38224
2,54,0.67187,0.01829,100,22,30,0.20168,0.36247
3,25,0.67223,0.01911,150,10,27,0.17125,0.32139
4,55,0.67235,0.02593,100,19,28,0.50538,0.39427


## 2. Elo on its own, for reference

Formula 1 used directly as a forecast, scored on 2014–2019. Every forecast
uses only earlier fights, so this never touches later years. This is the
table from the research page, and it is **not** how K is chosen: on its own,
Elo is best around K ≈ 63, while the model wants something faster.

In [3]:
pd.DataFrame([
    {"K": k, "elo_alone_log_loss": round(elo_forecast_log_loss(fights, k), 5)}
    for k in (16, 31.62, 63.1, 100, 158.5, 199.5, 316.2)
])

,K,elo_alone_log_loss
0,16.00,0.68804
1,31.62,0.68512
2,63.10,0.68265
3,100.00,0.68371
4,158.50,0.69244
5,199.50,0.70326
6,316.20,0.75373


## 3. Choose K by what helps the model

For every K on the grid (10 → 1000 in ~26% steps): attach Elo, take the
training years, score the model by expanding-window CV over 5 seeds. Expect a
broad plateau. Neighbouring K values within ~0.001 of each other are a tie,
not a ranking.

In [4]:
best_k, curve = choose_elo_k(without_elo, params_without)
save_elo_params(best_k, curve, seeds=range(5))

no_elo = cv_log_loss(train_without, params_without, seeds=range(5))["log_loss"]
k_table = pd.DataFrame(curve, columns=["K", "cv_log_loss"])
k_table["change_vs_no_elo"] = k_table["cv_log_loss"] - no_elo
print(f"without Elo: {no_elo:.5f}   chosen K: {best_k}")
k_table.round(5)

without Elo: 0.67188   chosen K: 316.2


,K,cv_log_loss,change_vs_no_elo
0,10.00,0.67147,-0.00040
1,12.59,0.67170,-0.00017
2,15.85,0.67192,0.00004
3,19.95,0.67158,-0.00030
4,25.12,0.67190,0.00003
5,31.62,0.67147,-0.00041
6,39.81,0.67113,-0.00074
7,50.12,0.67090,-0.00098
8,63.10,0.67100,-0.00088
9,79.43,0.66998,-0.00189


## 4. Rebuild the featured table with Elo

The same shared chain as notebook 03, now with the chosen K. Spot-check: UFC 69,
where Matt Serra upset Georges St-Pierre.

In [5]:
featured = build_featured_table(fights, fighters, alphas, elo_k=best_k)
save_processed(featured, "fights_featured")

assert manifest.unclassified_columns(featured) == []
print(f"{featured.shape[1]} columns, {len(manifest.feature_columns(featured))} features")
featured.loc[
    featured["event_date"].eq("2007-04-07") & featured["fighter_a_name"].eq("Matt Serra"),
    ["fighter_a_name", "fighter_b_name", *ELO_COLUMNS, "diff_prior_elo"],
].round(1)

373 columns, 108 features


,fighter_a_name,fighter_b_name,fighter_a_prior_elo,fighter_b_prior_elo,diff_prior_elo
631,Matt Serra,Georges St-Pierre,1671.1,2389.6,-718.5


## 5. Re-tune with Elo

The feature set changed, so the settings are re-chosen exactly the same way.
This saves `model_params.json` again, now for the table with Elo.

In [6]:
train, validation, _ = chronological_split(featured)      # the test years stay closed
study_with = tune(train, n_trials=60)
save_params(study_with)
params_with = study_with.best_params
print(f"best CV log loss, averaged over 3 seeds: {study_with.best_value:.5f}")
tuning_table(study_with, top=5)

best CV log loss, averaged over 3 seeds: 0.66643


,trial,log_loss,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features
0,57,0.66643,0.01030,400,10,7,27.02004,0.30642
1,56,0.66646,0.01243,400,12,10,22.76051,0.34139
2,50,0.66652,0.01230,400,12,10,21.86783,0.35290
3,53,0.66655,0.01284,450,13,7,24.05360,0.33889
4,54,0.66683,0.01093,450,17,7,27.11639,0.34782


## 6. The decision

Each version with its own tuned settings. First cross-validation over 8 seeds,
compared seed by seed. Then the one look at validation, with predictions
averaged over the same 8 seeds. The rule set before running: keep Elo if CV
improves in at least 7 of 8 seeds and validation doesn't contradict it.

In [7]:
SEEDS = range(8)
cv_without = cv_log_loss(train_without, params_without, seeds=SEEDS)
cv_with = cv_log_loss(train, params_with, seeds=SEEDS)
change = np.array(cv_with["log_loss_by_seed"]) - np.array(cv_without["log_loss_by_seed"])
print(f"CV log loss: without Elo {cv_without['log_loss']:.5f} -> with Elo {cv_with['log_loss']:.5f}")
print(f"change {change.mean():+.5f}, better in {(change < 0).sum()} of 8 seeds")

CV log loss: without Elo 0.67213 -> with Elo 0.66678
change -0.00536, better in 8 of 8 seeds


In [8]:
assert (validation_without["fights_url"].to_numpy() == validation["fights_url"].to_numpy()).all()

def seed_averaged(train_table, valid_table, params):
    """Validation predictions averaged over SEEDS, plus the seed-0 model."""
    X_tr, y_tr = manifest.make_xy(train_table)
    X_va, _ = manifest.make_xy(valid_table)
    models = [fit_model(X_tr, y_tr, params=params | {"random_state": s}) for s in SEEDS]
    return np.mean([predict_proba(m, X_va) for m in models], axis=0), models[0], X_va

p_without, _, _ = seed_averaged(train_without, validation_without, params_without)
p_with, model_with, X_val = seed_averaged(train, validation, params_with)
_, y_val = manifest.make_xy(validation)
mens = ~is_womens_fight(validation, history=featured).to_numpy()
y = np.asarray(y_val, dtype=bool)

result = compare(y[mens], p_with[mens], p_without[mens])
print(f"men's fights, with vs without Elo: fixes {result['fixes']}, breaks {result['breaks']}, "
      f"p = {result['p_value']:.3f}")
scorecard({"without Elo": p_without, "with Elo": p_with}, y_val, mens)

men's fights, with vs without Elo: fixes 53, breaks 51, p = 0.922


,men_accuracy,men_log_loss,men_auc,all_accuracy,all_log_loss,all_auc
model,,,,,,
without Elo,0.6366,0.6570,0.6743,0.6249,0.661,0.6598
with Elo,0.6392,0.6512,0.6777,0.6259,0.655,0.6671


In [9]:
# which feature families the tuned model with Elo relies on (men's validation fights)
groups = manifest.feature_groups(train)
group_importance(model_with, X_val[mens], y_val[mens], groups, repeats=20)

,group,columns,log_loss_rise,spread
0,physical,7,0.018757,0.003238
1,strength,1,0.016710,0.002684
2,defence,44,0.015994,0.002779
3,striking,32,0.004384,0.001375
4,grappling,10,0.003502,0.001144
5,record,8,0.001759,0.000685
6,cold_start,2,0.000838,0.000311
7,pace,2,0.000496,0.000149
8,damage,2,0.000116,0.000145
